# Heat Exchanger Network Design of a Four-Stream Problem
This notebook demonstrates the public heat exchanger network design workflow on a compact four-stream problem. It uses the problem-owned `design` accessor for direct use with `PinchProblem`.

## Imports

In [ ]:
import pandas as pd
from OpenPinch import PinchProblem

## Four-stream case
The stream and utility data are the converted four-stream heat exchanger network synthesis example used by the regression suite. The heat exchanger network design grid uses three approach temperatures, one derivative threshold, and one stage count so the notebook can show the top three networks without sending Couenne through a broad teaching sweep.

In [ ]:
four_stream_case = {
    "zone_tree": {
        "name": "Site",
        "type": "Site",
        "children": [
            {"name": "Process A", "type": "Process Zone"},
        ],
    },
    "options": {
        "COSTING_HX_AREA_EXP": 1.0,
        "COSTING_HX_UNIT_COST": 5500.0,
        "COSTING_HX_AREA_COEFF": 150.0,
        "HENS_APPROACH_TEMPERATURES": [10.0, 14.0, 18.0],
        "HENS_DERIVATIVE_THRESHOLDS": [0.5],
        "HENS_STAGE_SELECTION": [3],
        "HENS_BEST_SOLUTIONS_TO_SAVE": 3,
        "HENS_MAX_PARALLEL": 1,
        "HENS_OUTPUT_FORMATS": [],
        "HENS_RUN_ID": "four-stream-hen-demo",
    },
    "streams": [
        {
            "zone": "Site/Process A",
            "name": "Raw Milk",
            "t_supply": {"value": 650.0, "unit": "K"},
            "t_target": {"value": 370.0, "unit": "K"},
            "heat_flow": {"value": 2800.0, "unit": "kW"},
            "htc": {"value": 1.0, "unit": "kW/m^2/K"},
        },
        {
            "zone": "Site/Process A",
            "name": "HT Flash",
            "t_supply": {"value": 590.0, "unit": "K"},
            "t_target": {"value": 370.0, "unit": "K"},
            "heat_flow": {"value": 4400.0, "unit": "kW"},
            "htc": {"value": 1.0, "unit": "kW/m^2/K"},
        },
        {
            "zone": "Site/Process A",
            "name": "Milk Concentrate",
            "t_supply": {"value": 410.0, "unit": "K"},
            "t_target": {"value": 650.0, "unit": "K"},
            "heat_flow": {"value": 3600.0, "unit": "kW"},
            "htc": {"value": 1.0, "unit": "kW/m^2/K"},
        },
        {
            "zone": "Site/Process A",
            "name": "CIP Water",
            "t_supply": {"value": 350.0, "unit": "K"},
            "t_target": {"value": 500.0, "unit": "K"},
            "heat_flow": {"value": 1950.0, "unit": "kW"},
            "htc": {"value": 1.0, "unit": "kW/m^2/K"},
        },
    ],
    "utilities": [
        {
            "name": "HPS",
            "type": "Hot",
            "t_supply": {"value": 680.0, "unit": "K"},
            "t_target": {"value": 680.0, "unit": "K"},
            "htc": {"value": 5.0, "unit": "kW/m^2/K"},
            "price": {"value": 80.0, "unit": "$/MWh"},
        },
        {
            "name": "CW",
            "type": "Cold",
            "t_supply": {"value": 300.0, "unit": "K"},
            "t_target": {"value": 320.0, "unit": "K"},
            "htc": {"value": 1.0, "unit": "kW/m^2/K"},
            "price": {"value": 15.0, "unit": "$/MWh"},
        },
    ],
}

## Validate and prepare a problem
`PinchProblem` validates the dictionary payload and prepares the problem-owned `design` accessor used below.

In [ ]:
problem = PinchProblem(
    source=four_stream_case,
    project_name="four_stream_hen_design_service_demo",
)
validated = problem.validate()


pd.DataFrame(
    [
        {
            "project": problem.project_name,
            "streams": len(validated.streams),
            "utilities": len(validated.utilities or []),
            "approach_temperatures": validated.options["HENS_APPROACH_TEMPERATURES"],
            "derivative_thresholds": validated.options["HENS_DERIVATIVE_THRESHOLDS"],
            "stage_selection": validated.options["HENS_STAGE_SELECTION"],
        }
    ]
)

## Run the design workflow
The public service entry point is owned by the problem: `problem.design.heat_exchanger_network_synthesis()`. By default this call invokes the local solver-backed synthesis executor. Heat exchanger network configuration belongs in the case options loaded above; runtime options are reserved for target execution state such as `state_id`. Running this cell requires the `openpinch[synthesis]` Python extra and external solver binaries; `idaes get-extensions` installs Couenne and Ipopt into the IDAES extension directory, which OpenPinch will discover automatically.

In [ ]:
design = problem.design.heat_exchanger_network_synthesis()
top_ranked_network = design.ranked_networks[0]
network = design.network

pd.DataFrame(
    [
        {
            "run_id": design.run_id,
            "selected_rank": 1,
            "selected_method": design.method,
            "solver_name": design.solver_name,
            "solver_status": design.solver_status,
            "stage_count": design.stage_count,
            "ranked_network_count": len(design.ranked_networks),
            "exchanger_count": len(network.exchangers),
            "selected_network_is_rank_1": top_ranked_network.network == network,
            "cached_on_problem": problem.results.design == design,
        }
    ]
)

## Inspect the workflow manifest
The result carries serializable workflow metadata alongside the selected network.

In [ ]:
manifest = design.manifest

manifest_frame = pd.DataFrame(
    [
        {
            "run_id": manifest.run_id,
            "approach_temperatures": list(manifest.approach_temperatures),
            "derivative_thresholds": list(manifest.derivative_thresholds),
            "stage_selection": list(manifest.stage_selection),
            "task_ids": len(manifest.task_ids),
            "export_formats": list(manifest.export_formats),
        }
    ]
)

ranked_network_frame = pd.DataFrame(
    [
        {
            "method": outcome.task.method,
            "status": outcome.status,
            "approach_temperature_K": outcome.task.approach_temperature,
            "derivative_threshold": outcome.task.derivative_threshold,
            "stage_count": outcome.task.stage_count,
            "objective_value": outcome.objective_value,
        }
        for outcome in design.ranked_networks
    ]
)

display(manifest_frame)
ranked_network_frame

## Top network grid designs
Successful network candidates are exposed as `design.ranked_networks` and ranked by objective value after duplicate exchanger-connection structures are removed. `design.network` is the top-ranked network by default, and `design.select_network(solution_rank=...)` changes the selected network. The selected `HeatExchangerNetwork` draws itself with `design.network.build_grid_diagram(...)`. The Plotly grid diagrams below show the top three unique candidates from the accepted method family. Hover over exchanger markers to inspect match duty, area, and stream pairing.


In [ ]:
top_ranked_networks = design.get_n_best_networks(3)

ranked_network_summary = pd.DataFrame(
    [
        {
            "rank": rank,
            "method": outcome.task.method,
            "task_id": outcome.task.task_id,
            "approach_temperature_K": outcome.task.approach_temperature,
            "derivative_threshold": outcome.task.derivative_threshold,
            "stage_count": outcome.task.stage_count,
            "objective_value": outcome.objective_value,
            "recovery_duty_kW": outcome.network.total_duty(kind="recovery"),
            "hot_utility_duty_kW": outcome.network.total_duty(kind="hot_utility"),
            "cold_utility_duty_kW": outcome.network.total_duty(kind="cold_utility"),
        }
        for rank, outcome in enumerate(top_ranked_networks, start=1)
    ]
)
display(ranked_network_summary)

selected_network_rows = [
    {
        "selected_rank": 1,
        "task_id": design.task_id,
        "objective_value": design.ranked_networks[0].objective_value,
        "exchanger_count": len(design.network.exchangers),
    }
]
if len(top_ranked_networks) >= 2:
    rank_2_design = design.model_copy(deep=True).select_network(solution_rank=2)
    selected_network_rows.append(
        {
            "selected_rank": 2,
            "task_id": rank_2_design.task_id,
            "objective_value": rank_2_design.ranked_networks[1].objective_value,
            "exchanger_count": len(rank_2_design.network.exchangers),
        }
    )

pd.DataFrame(selected_network_rows)

In [ ]:
grid_diagrams = []

for rank in range(1, min(3, len(top_ranked_networks)) + 1):
    design.select_network(solution_rank=rank)
    diagram = design.network.build_grid_diagram()
    diagram.fig.update_layout(title_text=f"Rank {rank} heat exchanger network grid")
    grid_diagrams.append(diagram)
    display(diagram.fig)

design.select_network(solution_rank=1)
network = design.network

The returned diagram object wraps a Plotly figure. `diagram.save("network.png")` writes a static image through Kaleido, and `diagram.save("network.html")` writes an interactive Plotly document.


## Inspect the selected network
The selected `HeatExchangerNetwork` exposes ordered exchanger records plus stable labelled accessors for totals and stream-to-stream matches.

In [ ]:
exchanger_frame = pd.DataFrame(
    [
        {
            "id": exchanger.exchanger_id,
            "kind": exchanger.kind.value,
            "source": exchanger.source_stream,
            "sink": exchanger.sink_stream,
            "stage": exchanger.stage,
            "duty_kW": exchanger.duty,
            "area": exchanger.area,
            "active": exchanger.active,
            "match_allowed": exchanger.match_allowed,
            "source_outlet_K": exchanger.source_outlet_temperature,
            "sink_outlet_K": exchanger.sink_outlet_temperature,
        }
        for exchanger in network.exchangers
    ]
)

exchanger_frame

In [ ]:
first_recovery = next(
    exchanger
    for exchanger in network.exchangers
    if exchanger.kind == "recovery"
)
hot_utility_name = next(
    exchanger.source_stream
    for exchanger in network.exchangers
    if exchanger.kind == "hot_utility"
)
cold_utility_name = next(
    exchanger.sink_stream
    for exchanger in network.exchangers
    if exchanger.kind == "cold_utility"
)

pd.DataFrame(
    [
        {
            "metric": "total recovery duty",
            "value": problem.design.network.total_heat_recovery,
        },
        {
            "metric": "total hot utility duty",
            "value": problem.design.network.total_hot_utility,
        },
        {
            "metric": "total cold utility duty",
            "value": problem.design.network.total_cold_utility,
        },
        {
            "metric": f"{hot_utility_name} duty",
            "value": problem.design.network.utility(hot_utility_name),
        },
        {
            "metric": f"{cold_utility_name} duty",
            "value": problem.design.network.utility(cold_utility_name),
        },
        {
            "metric": "first recovery match duty",
            "value": network.labelled_value(
                "recovery_duty",
                source_stream=first_recovery.source_stream,
                sink_stream=first_recovery.sink_stream,
                stage=first_recovery.stage,
            ),
        },
    ]
)